In [1]:
!gdown 1N5V40HQsnCHHytdmG98YfXvN_3DCjV67
!tar -xf dataframe_hamshahri2.tar

import pandas as pd
import numpy as np
from typing import List, Dict, Tuple

df_docs = pd.read_csv('/content/hamshahri2/docs.csv.gz', compression='gzip')
df_judgments = pd.read_csv('/content/hamshahri2/judgments_dataframe.csv.gz', compression='gzip')
df_queries_fa = pd.read_csv('/content/hamshahri2/queries_fa.csv.gz', compression='gzip')

qrels = {}
for item in df_judgments[df_judgments['relevancy'] == 1].groupby('query_id')[['doc_id', 'relevancy']]:
    qrels[str(item[0])] = {val[0]: int(val[1]) for val in zip(item[1]['doc_id'], item[1]['relevancy'])}

print(f"Number of docs: {len(df_docs)}")
print(f"Number of queries: {len(df_queries_fa)}")
print(f"Number of qrels: {len(qrels)}")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class RankerTFIDF_Cosine:
    def __init__(self, docs) -> None:
        self.docs = docs
        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.vectorizer.fit_transform(self.docs)

    def batch_search(self, queries: List[str], k: int = 1000) -> Dict[str, List[Tuple[int, float]]]:
        query_vecs = self.vectorizer.transform(queries)
        sim_matrix = cosine_similarity(query_vecs, self.tfidf_matrix)

        results = {}
        for i, query in enumerate(queries):
            similarities = sim_matrix[i]
            top_indices = similarities.argsort()[-k:][::-1]
            results[query] = [
                (doc_id, float(similarities[doc_id]))
                for doc_id in top_indices
            ]
        return results


class RankerTFIDF_Proposed:
    def __init__(self, docs) -> None:
        self.docs = docs
        self.vectorizer = TfidfVectorizer(norm=None)
        self.tfidf_matrix = self.vectorizer.fit_transform(self.docs)

    def batch_search(self, queries: List[str], k: int = 1000) -> Dict[str, List[Tuple[int, float]]]:
        query_vecs = self.vectorizer.transform(queries)
        dot_matrix = (query_vecs @ self.tfidf_matrix.T).toarray()
        score_matrix = dot_matrix / (1.0 + dot_matrix)

        results = {}
        for i, query in enumerate(queries):
            scores = score_matrix[i]
            top_indices = scores.argsort()[-k:][::-1]
            results[query] = [
                (doc_id, float(scores[doc_id]))
                for doc_id in top_indices
            ]
        return results


docs = df_docs['TITLE'].tolist()
queries = list(df_queries_fa.TITLE.values)

print("Building Cosine model...")
scorer_cosine = RankerTFIDF_Cosine(docs)
results_cosine = scorer_cosine.batch_search(queries, k=1000)

print("Building Proposed model...")
scorer_proposed = RankerTFIDF_Proposed(docs)
results_proposed = scorer_proposed.batch_search(queries, k=1000)

print("Search finished.")

def convert_to_ranked_results(results, df_docs):
    ranked_results = {}
    for i, (query, doc_scores) in enumerate(results.items(), start=1):
        ranked_results[str(i)] = {
            df_docs.at[doc_id, 'DOCID']: float(score)
            for doc_id, score in doc_scores
        }
    return ranked_results

ranked_cosine = convert_to_ranked_results(results_cosine, df_docs)
ranked_proposed = convert_to_ranked_results(results_proposed, df_docs)

!pip install -q beir

from beir.retrieval.evaluation import EvaluateRetrieval

top_k_values = [1, 3, 5, 10, 100, 1000]

print("\n=== Cosine Similarity Results ===")
metrics_cosine = EvaluateRetrieval.evaluate(qrels, ranked_cosine, top_k_values)

print("\n=== Proposed Method Results ===")
metrics_proposed = EvaluateRetrieval.evaluate(qrels, ranked_proposed, top_k_values)

def get_metrics_dataframe(metrics):
    data = []
    for metric_group in metrics:
        for metric_name, score in metric_group.items():
            metric, k = metric_name.split('@')
            data.append({
                'Metric': metric,
                'k': int(k),
                'Score': score
            })
    df = pd.DataFrame(data)
    return df.pivot(index='k', columns='Metric', values='Score').reset_index()

df_cosine = get_metrics_dataframe(metrics_cosine)
df_proposed = get_metrics_dataframe(metrics_proposed)

print("\nCosine Similarity:")
display(df_cosine)

print("\nProposed Method:")
display(df_proposed)

comparison = df_cosine.merge(df_proposed, on='k', suffixes=('_Cosine', '_Proposed'))
print("\nComparison:")
display(comparison)

df_cosine.to_csv('metrics_cosine.csv', index=False)
df_proposed.to_csv('metrics_proposed.csv', index=False)
comparison.to_csv('metrics_comparison.csv', index=False)

print("\nSaved files:")
print(" - metrics_cosine.csv")
print(" - metrics_proposed.csv")
print(" - metrics_comparison.csv")

Downloading...
From (original): https://drive.google.com/uc?id=1N5V40HQsnCHHytdmG98YfXvN_3DCjV67
From (redirected): https://drive.google.com/uc?id=1N5V40HQsnCHHytdmG98YfXvN_3DCjV67&confirm=t&uuid=6a92a7d1-156a-4839-b7c3-ddeb0768eb51
To: /content/dataframe_hamshahri2.tar
100% 160M/160M [00:02<00:00, 59.1MB/s]
Number of docs: 318624
Number of queries: 50
Number of qrels: 50
Building Cosine model...
Building Proposed model...
Search finished.

=== Cosine Similarity Results ===

=== Proposed Method Results ===

Cosine Similarity:


Metric,k,MAP,NDCG,P,Recall
0,1,0.00641,0.48000,0.48000,0.00641
1,3,0.01711,0.52614,0.53333,0.02006
2,5,0.02576,0.53291,0.54000,0.03270
3,10,0.04043,0.49620,0.48400,0.05555
4,100,0.11300,0.31309,0.22160,0.22210
5,1000,0.14576,0.40397,0.05000,0.46653



Proposed Method:


Metric,k,MAP,NDCG,P,Recall
0,1,0.00477,0.44000,0.44000,0.00477
1,3,0.00996,0.36816,0.34667,0.01184
2,5,0.01391,0.35540,0.33600,0.01831
3,10,0.02343,0.34582,0.33200,0.03758
4,100,0.08435,0.27266,0.20940,0.20735
5,1000,0.12065,0.37860,0.05124,0.46859



Comparison:


Metric,k,MAP_Cosine,NDCG_Cosine,P_Cosine,Recall_Cosine,MAP_Proposed,NDCG_Proposed,P_Proposed,Recall_Proposed
0,1,0.00641,0.48000,0.48000,0.00641,0.00477,0.44000,0.44000,0.00477
1,3,0.01711,0.52614,0.53333,0.02006,0.00996,0.36816,0.34667,0.01184
2,5,0.02576,0.53291,0.54000,0.03270,0.01391,0.35540,0.33600,0.01831
3,10,0.04043,0.49620,0.48400,0.05555,0.02343,0.34582,0.33200,0.03758
4,100,0.11300,0.31309,0.22160,0.22210,0.08435,0.27266,0.20940,0.20735
5,1000,0.14576,0.40397,0.05000,0.46653,0.12065,0.37860,0.05124,0.46859



Saved files:
 - metrics_cosine.csv
 - metrics_proposed.csv
 - metrics_comparison.csv
